# GlucoTwin — Couche 2 : prévision glycémique évaluée honnêtement**AI for Health (PGE5, Prof. A. Kar)** · Regis Likassi · Hakim Djomo · Jean Direl Nze · Xavier Ondo · Seth Ndinga---## La thèse> À 30 minutes, la prévision glycémique est **saturée** : la persistance — « la glycémie ne bougera pas » — est quasi imbattable. La question utile n'est donc pas *quel modèle a la plus petite erreur*, mais **où** la prévision a une valeur clinique, et **sait-on quand elle est fiable ?**## Ce que ce notebook teste| | Hypothèse ||---|---|| **H1** | L'avantage sur la persistance **croît avec l'horizon** || **H2** | Les **concepts métaboliques** apportent quelque chose que l'historique glycémique seul n'a pas || **H3** | La MAE **masque** la valeur clinique (détection des événements) || **H4** | On peut produire des intervalles à **couverture garantie** |## L'architecture testée```emploi du temps ──► COUCHE 1 : concepts métaboliques ──► COUCHE 2 : prévision                    (physiologie, NON apprise)            (apprise, ici)```La couche 1 n'est pas entraînée : METs, équations de Frayn, cinétique d'absorption, circadien, phénomène de l'aube. Elle joue le rôle de **goulot d'étranglement interprétable** — l'idée des *Concept Bottleneck Models* (Koh et al., ICML 2020).> ⚠️ **Les données de ce notebook sont synthétiques.** Aucun patient réel. Les résultats valident **le logiciel et le protocole**, pas la physiologie. La section finale explique comment brancher CGMacros.

## 1. InstallationTrois façons de charger le code sur Kaggle. La cellule suivante les essaie dans l'ordre et s'arrête à la première qui marche.

In [ ]:
import subprocess, sys, pathlib, importlibREPO = "https://github.com/VOTRE-COMPTE/GlucoTwin.git"   # <-- à personnaliserdef _ok():    try:        importlib.import_module("glucotwin"); return True    except ImportError:        return Falseif not _ok():    # (a) le dépôt est déjà présent (dossier local ou Kaggle Dataset)    for base in [pathlib.Path.cwd(), pathlib.Path("/kaggle/input")]:        if not base.exists():            continue        for cand in list(base.glob("**/src/glucotwin"))[:1]:            sys.path.insert(0, str(cand.parent))    # (b) sinon, cloner    if not _ok() and "VOTRE-COMPTE" not in REPO:        subprocess.run(["git", "clone", "-q", REPO, "/kaggle/working/GlucoTwin"], check=False)        sys.path.insert(0, "/kaggle/working/GlucoTwin/src")assert _ok(), (    "glucotwin introuvable.\n"    "  • renseignez REPO ci-dessus, ou\n"    "  • ajoutez le dépôt en Kaggle Dataset (Add Data), ou\n"    "  • lancez ce notebook depuis la racine du dépôt.")import glucotwinprint("glucotwin", glucotwin.__version__, "|", pathlib.Path(glucotwin.__file__).parent)

In [ ]:
import json, time, warningsimport numpy as npimport matplotlib.pyplot as pltfrom glucotwin.layer2.cohort import build_cohortfrom glucotwin.layer2.features import build_features, CONCEPT_COLSfrom glucotwin.layer2.evaluation import lopo_evaluate, print_reportfrom glucotwin.layer2.models import model_zoowarnings.filterwarnings("ignore", category=UserWarning)# ---------------------------------------------------------------- réglagesQUICK = True          # True = ~2 min (mise au point) · False = run completif QUICK:    N_PATIENTS, N_DAYS, HORIZONS, MAX_FOLDS = 14, 4, [30, 60, 120], 10else:    N_PATIENTS, N_DAYS, HORIZONS, MAX_FOLDS = 45, 8, [30, 60, 90, 120], NoneSEED, ALPHA = 7, 0.10           # ALPHA=0.10 -> intervalles à 90 %OUT = pathlib.Path("/kaggle/working") if pathlib.Path("/kaggle/working").exists() else pathlib.Path(".")print(f"mode {'RAPIDE' if QUICK else 'COMPLET'} | {N_PATIENTS} patients x {N_DAYS} jours "      f"| horizons {HORIZONS} | sorties -> {OUT}")if QUICK:    print("\n⚠️  Le mode RAPIDE sert à la mise au point. Avec une dizaine de patients,")    print("    la variabilité inter-patients domine : H1 peut ressortir NON CONFIRMÉE")    print("    par simple manque de puissance statistique. Pour conclure, passez")    print("    QUICK = False (compter 20 à 40 min).")

In [ ]:
# Palette validée (contraste et daltonisme vérifiés par script)BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"RAMP = ["#86b6ef", "#5598e7", "#2a78d6", "#184f95"]      # ordinal : de moins à plusGRID, INK, MUTED = "#e1e0d9", "#0b0b0b", "#898781"GOOD = "#0ca30c"plt.rcParams.update({    "figure.dpi": 130, "font.size": 10,    "axes.spines.top": False, "axes.spines.right": False,    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK, "text.color": INK,    "xtick.color": MUTED, "ytick.color": MUTED,    "grid.color": GRID, "grid.linewidth": 0.8,    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.bbox": "tight",})def save(fig, name):    fig.savefig(OUT / f"{name}.png", dpi=150)    print(f"  figure -> {OUT/f'{name}.png'}")

## 2. DonnéesCohorte virtuelle **volontairement non circulaire** : la glycémie n'est pas une fonction directe des concepts. Elle est intégrée dans le temps avec une **clairance non linéaire** absente des concepts, des **collations non déclarées** (plancher d'erreur irréductible) et du **bruit de capteur**. Le modèle doit donc réellement apprendre.

In [ ]:
t0 = time.time()df = build_cohort(n_patients=N_PATIENTS, days_per_patient=N_DAYS, seed=SEED)print(f"{len(df):,} pas de 5 min | {df.patient.nunique()} patients | {time.time()-t0:.0f}s")print(f"glycémie moyenne {df.glucose.mean():.0f} mg/dL")print(f"temps dans la cible 70-180 : {(df.glucose.between(70,180)).mean()*100:.0f} %")print(f"hypo <70 : {(df.glucose<70).mean()*100:.2f} %  |  hyper >180 : {(df.glucose>180).mean()*100:.2f} %")fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))ax[0].hist(df.glucose, bins=60, color=BLUE, alpha=.85)ax[0].axvline(70, color=ORANGE, ls="--", lw=1.5); ax[0].axvline(180, color=ORANGE, ls="--", lw=1.5)ax[0].set_xlabel("glycémie (mg/dL)"); ax[0].set_ylabel("effectif")ax[0].set_title("Distribution sur la cohorte", loc="left")par_pat = df.groupby("patient").glucose.mean().sort_values()ax[1].barh(range(len(par_pat)), par_pat.values, color=BLUE, height=.8)ax[1].axvline(140, color=MUTED, ls=":", lw=1)ax[1].set_yticks([]); ax[1].set_xlabel("glycémie moyenne (mg/dL)")ax[1].set_title("Un patient par barre : la cohorte est hétérogène", loc="left")for a in ax: a.grid(axis="y", alpha=.4)plt.tight_layout(); save(fig, "00_cohorte"); plt.show()

In [ ]:
# Une journée : de l'emploi du temps à la glycémied = df[(df.patient == df.patient.iloc[0]) & (df.day == 0)]fig, ax = plt.subplots(4, 1, figsize=(9.5, 7.5), sharex=True,                       gridspec_kw={"height_ratios": [2.2, 1, 1, 1]})ax[0].axhspan(70, 180, color=GOOD, alpha=.07, lw=0)ax[0].plot(d.t_h, d.glucose, color=BLUE, lw=2)ax[0].set_ylabel("glycémie\n(mg/dL)")ax[0].set_title("Une journée du jumeau — les concepts expliquent la courbe", loc="left")ax[1].fill_between(d.t_h, d.carb_ra_g_min*1000, color=ORANGE, alpha=.3, lw=0)ax[1].plot(d.t_h, d.carb_ra_g_min*1000, color=ORANGE, lw=1.6)ax[1].set_ylabel("apport\nglucides\n(mg/min)")ax[2].fill_between(d.t_h, d.glucose_uptake_mg_min, color=AQUA, alpha=.3, lw=0)ax[2].plot(d.t_h, d.glucose_uptake_mg_min, color=AQUA, lw=1.6)ax[2].set_ylabel("captation\n(mg/min)")ax[3].plot(d.t_h, d.net_glucose_flux_mg_min, color=BLUE, lw=1.6)ax[3].axhline(0, color=MUTED, lw=.8, ls="--")ax[3].fill_between(d.t_h, 0, d.net_glucose_flux_mg_min,                   where=d.net_glucose_flux_mg_min > 0, color=ORANGE, alpha=.25)ax[3].fill_between(d.t_h, 0, d.net_glucose_flux_mg_min,                   where=d.net_glucose_flux_mg_min < 0, color=AQUA, alpha=.25)ax[3].set_ylabel("flux net\n(mg/min)"); ax[3].set_xlabel("heure")ax[3].set_xticks(range(0, 25, 3))for a in ax: a.grid(axis="y", alpha=.45)plt.tight_layout(); save(fig, "01_journee"); plt.show()print("Flux net > 0 : la glycémie monte (digestion) · < 0 : elle descend (effort)")

## 3. H1 — L'avantage croît-il avec l'horizon ?**Protocole, non négociable :**- **leave-one-patient-out** — chaque patient est testé sans jamais avoir été vu à l'entraînement ;- **baseline de persistance** systématique ;- **cible = variation**, pas niveau — sinon le modèle recopie la valeur actuelle et paraît excellent ;- **test apparié** (Wilcoxon) + intervalle de confiance sur le gain.

In [ ]:
zoo = model_zoo()reports, timings = {}, {}for h in HORIZONS:    X, y, groups, g_now, names = build_features(df, horizon_min=h, target="delta")    t = time.time()    reports[h] = lopo_evaluate(X, y, groups, g_now, zoo["hgb"], target="delta",                               alpha=ALPHA, max_patients=MAX_FOLDS, seed=SEED)    timings[h] = time.time() - t    s = reports[h].summary()    print(f"horizon {h:>3} min | {X.shape[0]:>7,} ex. | MAE {s['mae_model']:5.2f} "          f"vs pers. {s['mae_persistence']:5.2f} | gain {s['gain_mae']:+5.2f} "          f"| p={s['p_value']:.1e} | {timings[h]:.0f}s")

In [ ]:
print_report(reports[HORIZONS[-1]], f"Détail — horizon {HORIZONS[-1]} min")

In [ ]:
# FIGURE CENTRALES = {h: reports[h].summary() for h in HORIZONS}gains = [S[h]["gain_mae"] for h in HORIZONS]errs  = [(S[h]["ic95_gain"][1] - S[h]["ic95_gain"][0]) / 2 for h in HORIZONS]fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))ax[0].plot(HORIZONS, [S[h]["mae_persistence"] for h in HORIZONS], "o--",           color=ORANGE, lw=2, ms=7, label="persistance")ax[0].plot(HORIZONS, [S[h]["mae_model"] for h in HORIZONS], "o-",           color=BLUE, lw=2, ms=7, label="modèle")ax[0].set_xlabel("horizon (min)"); ax[0].set_ylabel("MAE (mg/dL)")ax[0].set_title("L'écart se creuse", loc="left"); ax[0].legend(frameon=False)ax[1].errorbar(HORIZONS, gains, yerr=errs, fmt="o-", color=BLUE, lw=2, ms=7,               capsize=4, ecolor=MUTED)ax[1].axhline(0, color=ORANGE, lw=1.5, ls="--")ax[1].set_xlabel("horizon (min)"); ax[1].set_ylabel("gain sur la persistance (mg/dL)")ax[1].set_title("Le modèle ne sert qu'au-delà du court terme", loc="left")# Distribution patient par patient — l'honnêteté est ici, pas dans la moyennedata = [reports[h].mae_persistence - reports[h].mae_model for h in HORIZONS]bp = ax[2].boxplot(data, positions=range(len(HORIZONS)), widths=.55,                   patch_artist=True, medianprops=dict(color=INK, lw=1.5))for patch, c in zip(bp["boxes"], RAMP):    patch.set_facecolor(c); patch.set_alpha(.75); patch.set_edgecolor("none")for i, dd in enumerate(data):    ax[2].scatter(np.full(len(dd), i) + np.random.default_rng(0).normal(0, .05, len(dd)),                  dd, s=9, color=INK, alpha=.35, zorder=3)ax[2].axhline(0, color=ORANGE, lw=1.5, ls="--")ax[2].set_xticks(range(len(HORIZONS))); ax[2].set_xticklabels(HORIZONS)ax[2].set_xlabel("horizon (min)"); ax[2].set_ylabel("gain par patient (mg/dL)")ax[2].set_title("Un point = un patient", loc="left")for a in ax:    a.grid(axis="y", alpha=.45)    if a is not ax[2]: a.set_xticks(HORIZONS)plt.tight_layout(); save(fig, "02_horizons"); plt.show()print(f"{'horizon':>8} {'gain':>8} {'IC95':>20} {'p':>10} {'patients gagnés':>16}")for h in HORIZONS:    lo, hi = S[h]["ic95_gain"]    print(f"{h:>8} {S[h]['gain_mae']:>+8.2f} {f'[{lo:+.2f}, {hi:+.2f}]':>20} "          f"{S[h]['p_value']:>10.1e} {S[h]['patients_gagnes']:>10}/{S[h]['n_patients']}")

## 4. H2 — Les concepts métaboliques servent-ils à quelque chose ?**C'est l'expérience qui justifie toute la couche 1.** Si l'historique glycémique seul fait aussi bien, le goulot métabolique est une élégance inutile.On empile progressivement les groupes de features :| Jeu | Contenu ||---|---|| `historique` | glycémie actuelle, retards, variations, vitesse, accélération, heure, poids || `+ repas` | glucides en cours d'absorption, débit d'apparition || `+ activité` | METs, dépense, oxydation, captation, déficit glycogénique || `+ modulateurs` | circadien, sensibilité à l'insuline, aube, production hépatique, flux net |

In [ ]:
GROUPES = {    "historique":     [],    "+ repas":        ["cob_g", "carb_ra_g_min"],    "+ activité":     ["cob_g", "carb_ra_g_min", "asleep", "met_now",                       "energy_rate_kcal_min", "cho_ox_rate_g_min",                       "glucose_uptake_mg_min", "glycogen_deficit_g"],    "+ modulateurs":  CONCEPT_COLS,      # tout}H_ABL = HORIZONS[-1]         # l'horizon où le modèle a le plus de valeurablation = {}for label, cols in GROUPES.items():    X, y, groups, g_now, names = build_features(df, horizon_min=H_ABL,                                                target="delta", concept_cols=cols)    rep = lopo_evaluate(X, y, groups, g_now, zoo["hgb"], target="delta",                        alpha=ALPHA, max_patients=MAX_FOLDS, seed=SEED)    ablation[label] = rep    s = rep.summary()    print(f"{label:<16} {X.shape[1]:>3} features | MAE {s['mae_model']:5.2f} "          f"| gain {s['gain_mae']:+5.2f} mg/dL")

In [ ]:
labels = list(GROUPES)maes   = [ablation[l].summary()["mae_model"] for l in labels]gains_a = [ablation[l].summary()["gain_mae"] for l in labels]base_mae = maes[0]fig, ax = plt.subplots(1, 2, figsize=(10.5, 3.9))ax[0].bar(range(len(labels)), maes, color=RAMP, width=.66)ax[0].axhline(base_mae, color=MUTED, ls=":", lw=1.2)ax[0].set_xticks(range(len(labels))); ax[0].set_xticklabels(labels, rotation=15, ha="right")ax[0].set_ylabel("MAE (mg/dL)")ax[0].set_title(f"Chaque groupe de concepts ajoute-t-il ? (horizon {H_ABL} min)", loc="left")ax[0].set_ylim(min(maes)*0.94, max(maes)*1.03)for i, v in enumerate(maes):    ax[0].text(i, v, f"{v:.2f}", ha="center", va="bottom", fontsize=9)reduction = [(base_mae - m) / base_mae * 100 for m in maes]ax[1].plot(range(len(labels)), reduction, "o-", color=BLUE, lw=2, ms=8)ax[1].axhline(0, color=ORANGE, ls="--", lw=1.5)ax[1].set_xticks(range(len(labels))); ax[1].set_xticklabels(labels, rotation=15, ha="right")ax[1].set_ylabel("réduction de MAE vs historique seul (%)")ax[1].set_title("Apport cumulé des concepts", loc="left")for a in ax: a.grid(axis="y", alpha=.45)plt.tight_layout(); save(fig, "03_ablation"); plt.show()delta_total = base_mae - maes[-1]print(f"Historique seul : MAE {base_mae:.2f} mg/dL")print(f"Avec tous les concepts : MAE {maes[-1]:.2f} mg/dL")print(f"=> les concepts métaboliques valent {delta_total:+.2f} mg/dL "      f"({delta_total/base_mae*100:+.1f} %)")print("\nSi cet écart est proche de zéro, la couche 1 n'apporte rien EN PRÉCISION —")print("mais elle garderait sa valeur en INTERPRÉTABILITÉ. Les deux se discutent séparément.")

## 5. Quels concepts comptent ?L'importance par permutation mesure la dégradation quand on casse une variable. Contrairement à SHAP, elle est **indépendante du modèle** et directement interprétable.

In [ ]:
from sklearn.inspection import permutation_importanceX, y, groups, g_now, names = build_features(df, horizon_min=H_ABL, target="delta")pats = list(dict.fromkeys(groups))te = np.isin(groups, pats[:max(2, len(pats)//5)])          # patients de testmodel = zoo["hgb"](); model.fit(X[~te], y[~te])imp = permutation_importance(model, X[te], y[te], n_repeats=5,                             random_state=SEED, scoring="neg_mean_absolute_error")order = np.argsort(imp.importances_mean)[-14:]fig, ax = plt.subplots(figsize=(7.5, 5))cols = [ORANGE if names[i] in CONCEPT_COLS else BLUE for i in order]ax.barh(range(len(order)), imp.importances_mean[order],        xerr=imp.importances_std[order], color=cols, height=.72,        error_kw=dict(ecolor=MUTED, lw=1))ax.set_yticks(range(len(order))); ax.set_yticklabels([names[i] for i in order], fontsize=9)ax.set_xlabel("dégradation de la MAE si la variable est cassée (mg/dL)")ax.set_title("Ce sur quoi le modèle s'appuie vraiment", loc="left")ax.grid(axis="x", alpha=.45)handles = [plt.Rectangle((0,0),1,1,color=ORANGE), plt.Rectangle((0,0),1,1,color=BLUE)]ax.legend(handles, ["concept métabolique (couche 1)", "historique glycémique"],          frameon=False, loc="lower right", fontsize=9)plt.tight_layout(); save(fig, "04_importance"); plt.show()

## 6. H3 — La MAE dit-elle la même chose que la clinique ?Une erreur moyenne excellente peut coexister avec une incapacité totale à annoncer une hyperglycémie : les événements sont **rares**, donc noyés dans une moyenne.*(Deux échelles différentes → deux panneaux, jamais deux axes y sur un même graphique.)*

In [ ]:
sens_hyper, sens_hypo, ratio = [], [], []for h in HORIZONS:    c = reports[h].clinical()    sens_hyper.append(c["hyper"]["sensibilite"]*100 if c["hyper"]["n_events"] else np.nan)    sens_hypo.append(c["hypo"]["sensibilite"]*100 if c["hypo"]["n_events"] else np.nan)    ratio.append(reports[h].y_pred_abs.std() / reports[h].y_true_abs.std())fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))ax[0].plot(HORIZONS, gains, "o-", color=BLUE, lw=2, ms=8)ax[0].axhline(0, color=MUTED, ls=":", lw=1)ax[0].set_xlabel("horizon (min)"); ax[0].set_ylabel("gain MAE (mg/dL)")ax[0].set_title("① La MAE dit : ça s'améliore", loc="left")ax[1].plot(HORIZONS, sens_hyper, "s-", color=ORANGE, lw=2, ms=8)ax[1].set_xlabel("horizon (min)"); ax[1].set_ylabel("sensibilité (%)")ax[1].set_ylim(0, 100)ax[1].set_title("② La clinique dit : ça se dégrade", loc="left")ax[2].plot(HORIZONS, ratio, "o-", color=AQUA, lw=2, ms=8)ax[2].axhline(1.0, color=MUTED, ls="--", lw=1.2)ax[2].set_xlabel("horizon (min)"); ax[2].set_ylabel("σ prédictions / σ réel")ax[2].set_ylim(0, 1.15)ax[2].set_title("③ Pourquoi : régression vers la moyenne", loc="left")for a in ax: a.grid(axis="y", alpha=.45); a.set_xticks(HORIZONS)plt.tight_layout(); save(fig, "05_mae_vs_clinique"); plt.show()print(f"{'horizon':>8} {'gain MAE':>10} {'sensib. hyper':>15} {'ratio σ':>10}")for i, h in enumerate(HORIZONS):    print(f"{h:>8} {gains[i]:>+10.2f} {sens_hyper[i]:>14.0f}% {ratio[i]:>10.2f}")print("\nUn ratio < 1 = le modèle se réfugie dans la moyenne : il n'ose plus annoncer")print("les extrêmes, donc il rate les événements — alors même que sa MAE s'améliore.")print("C'est la démonstration que la MAE seule ne suffit pas à juger un jumeau.")

In [ ]:
# L'erreur n'a pas le même prix selon la zone glycémiquezn = [z for z in reports[HORIZONS[0]].clinical()["zones"]]fig, ax = plt.subplots(figsize=(8, 3.9))w = 0.8 / len(HORIZONS)for i, h in enumerate(HORIZONS):    z = reports[h].clinical()["zones"]    vals = [z[k]["mae"] if z[k]["n"] > 30 else np.nan for k in zn]    ax.bar(np.arange(len(zn)) + i*w, vals, w*0.9, color=RAMP[i % len(RAMP)], label=f"{h} min")ax.set_xticks(np.arange(len(zn)) + 0.4 - w/2)ax.set_xticklabels(zn, rotation=15, ha="right")ax.set_ylabel("MAE (mg/dL)")ax.set_title("L'erreur explose là où elle est dangereuse", loc="left")ax.legend(frameon=False, fontsize=9); ax.grid(axis="y", alpha=.45)plt.tight_layout(); save(fig, "06_zones"); plt.show()

## 7. H4 — Les intervalles tiennent-ils leur promesse ?La **prédiction conforme** garantit une couverture ≥ 1−α *sans hypothèse* sur la distribution des erreurs. On vérifie empiriquement.

In [ ]:
cov  = [S[h]["couverture"]*100 for h in HORIZONS]wide = [S[h]["largeur_intervalle"] for h in HORIZONS]fig, ax = plt.subplots(1, 2, figsize=(10, 3.7))ax[0].plot(HORIZONS, cov, "o-", color=BLUE, lw=2, ms=8)ax[0].axhline((1-ALPHA)*100, color=ORANGE, lw=1.5, ls="--")ax[0].annotate(f"cible {(1-ALPHA)*100:.0f} %", (HORIZONS[0], (1-ALPHA)*100),               textcoords="offset points", xytext=(4, 6), color=ORANGE, fontsize=9)ax[0].set_ylim(70, 100); ax[0].set_xlabel("horizon (min)")ax[0].set_ylabel("couverture observée (%)")ax[0].set_title("Couverture des intervalles", loc="left")ax[1].plot(HORIZONS, wide, "o-", color=AQUA, lw=2, ms=8)ax[1].set_xlabel("horizon (min)"); ax[1].set_ylabel("largeur moyenne (mg/dL)")ax[1].set_title("Le modèle avoue son incertitude", loc="left")for a in ax: a.grid(axis="y", alpha=.45); a.set_xticks(HORIZONS)plt.tight_layout(); save(fig, "07_conforme"); plt.show()for h, c, w_ in zip(HORIZONS, cov, wide):    print(f"  {h:>3} min : couverture {c:5.1f} %  |  largeur {w_:5.0f} mg/dL")print(f"\nSous-couverture attendue : la garantie conforme suppose des données")print("échangeables, or on calibre sur certains patients et on teste sur un patient")print("JAMAIS VU. Piste : Mondrian conformal, ou calibration sur les premiers jours")print("du patient lui-même. À reporter tel quel plutôt qu'à masquer.")

## 8. Synthèse

In [ ]:
resume = {    "config": {"quick": QUICK, "patients": N_PATIENTS, "jours": N_DAYS,               "horizons": HORIZONS, "alpha": ALPHA, "seed": SEED},    "horizons": {str(h): S[h] for h in HORIZONS},    "ablation": {k: v.summary() for k, v in ablation.items()},    "clinique": {str(h): {"sensibilite_hyper": sens_hyper[i], "ratio_sigma": ratio[i]}                 for i, h in enumerate(HORIZONS)},}(OUT / "resultats.json").write_text(json.dumps(resume, indent=2, ensure_ascii=False,                                               default=float), encoding="utf-8")print("=" * 72)print(f"{'HYPOTHÈSE':<58}{'VERDICT':>14}")print("=" * 72)h1 = gains[-1] > gains[0] and S[HORIZONS[-1]]["p_value"] < 0.05h2 = (base_mae - maes[-1]) / base_mae > 0.02h3 = (np.nanmax(sens_hyper) - np.nanmin(sens_hyper)) > 5 or ratio[-1] < 0.9h4 = abs(np.mean(cov) - (1-ALPHA)*100) < 10for txt, ok in [    ("H1 — l'avantage croît avec l'horizon", h1),    ("H2 — les concepts métaboliques apportent en précision", h2),    ("H3 — la MAE masque la valeur clinique", h3),    ("H4 — les intervalles couvrent approximativement leur cible", h4),]:    print(f"{txt:<58}{'CONFIRMÉE' if ok else 'NON CONFIRMÉE':>14}")print("=" * 72)if QUICK and not h1:    print("\n→ H1 non confirmée en mode RAPIDE : c'est attendu. Trop peu de patients")    print("  pour que le test apparié atteigne le seuil. Relancez avec QUICK = False.")print(f"\nArtefacts écrits dans {OUT} : 8 figures + resultats.json")print("\n⚠️  RAPPEL : cohorte SYNTHÉTIQUE. Ces résultats valident le logiciel et le")print("    protocole, pas la physiologie. Aucun patient réel n'a été utilisé.")

## 9. Brancher les vraies données (CGMacros)**Une seule cellule change** : celle qui construit `df`. Tout le reste du notebook tourne à l'identique — c'est tout l'intérêt d'avoir figé le protocole avant de voir les données.Il faut produire un `DataFrame` de même forme :| colonne | contenu ||---|---|| `patient`, `day`, `t_h` | identifiants et temps || `glucose` | CGM Dexcom (mg/dL) || `weight_kg` | anthropométrie || les 13 concepts | sortie de la couche 1 |```python# 1. horaires des repas + macronutriments        -> objets Meal# 2. série de METs Fitbit (mesurés, pas le catalogue)# 3. glucotwin.day_concepts.compute_day_concepts  -> concepts alignés sur le CGM# 4. CGM Dexcom                                   -> colonne glucose# df = build_cgmacros_concepts("/kaggle/input/cgmacros")```**Sur Kaggle :** créez un Dataset privé avec l'archive PhysioNet (627 Mo, CC BY-NC-SA 4.0 — ne la rendez pas publique), puis pointez le chemin ci-dessus.### Ce qui changera probablement- la **détection d'événements** sera plus basse (vraies hypoglycémies, plus rares et plus brutales) ;- le **gain sur la persistance** sera plus faible qu'ici — le monde réel est plus bruité ;- l'**ablation** deviendra l'expérience décisive : si les concepts n'apportent rien sur données réelles, c'est un résultat publiable en soi.> Ne réglez rien après avoir vu les résultats réels. Le protocole est figé ; c'est ce qui rendra vos conclusions crédibles.